In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import pandas as pd
eagles_df = pd.read_csv('../../data/EAGLE_S_PositionReport_5min_25Dec2024_26Dec2024.csv')
vezhen_df = pd.read_csv('../../data/VEZHEN_PositionReport_5min_25Jan2025_26Jan2025.csv')

In [3]:
raw_df = pd.concat([eagles_df, vezhen_df])

In [4]:
from datetime import timedelta, date

from itertools import product
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import cKDTree

In [5]:
import sys
sys.path.append('../..')

from src.data_preparation_eagles import (
    LON_MIN,
    LON_MAX,
    LAT_MIN,
    LAT_MAX,

    STEP_BACK_DEFAULT,
    STEP_FORW_DEFAULT,
    ROLLING_WINDOW_SIZE,

    DOWN_SAMPLE_MINUTES,
    GAP_LIMIT,
    ANGLE_COLS,

    merge_with_meteo,
    feature_engineering
)

SLIDINGW_STEPS=5
N_OUT_FEATS=3
N_INP_FEATS=20

qrf


In [6]:
# Data preparation adapted from 1data_preaparation_eagles.py

meteo_df = pd.read_csv(
    '../../data/weather_df_eagles_vezhen.csv',
    parse_dates=['time']
)

latitudes = meteo_df['Latitude'].unique()
longitudes = meteo_df['Longitude'].unique()
lat_lon_grid = list(product(latitudes, longitudes))
lat_lon_tree = cKDTree(lat_lon_grid)

meteo_df = meteo_df\
    .set_index( ['time', 'Latitude', 'Longitude'])\
    .drop(columns=['Unnamed: 0'])

# COG and Heading limit
raw_df = raw_df[raw_df['COURSE'] >= 0.0]
raw_df = raw_df[raw_df['HEADING'] >= 0.0]
raw_df = raw_df[raw_df['COURSE'] <= 360.0]
raw_df = raw_df[raw_df['HEADING'] <= 360.0]

raw_df.loc[raw_df['SPEED'] > 25, 'SPEED'] = np.nan

# Keep only underway data
# Moored > 1 knot is considered underway
nav_stat_mapping = {
    'Under way using engine': 0,
    'Under way sailing': 1,
    'Not under command': 2,
    'Moored': 5,
    'Under way sailing': 8
}
raw_df.loc[
    (raw_df['NAVSTAT'] == nav_stat_mapping['Moored']) &
    (raw_df['SPEED'] < 1)
] = np.nan

# Drop anyone not underway
raw_df = raw_df[
    (raw_df['NAVSTAT'] == nav_stat_mapping['Under way using engine']) |
    (raw_df['NAVSTAT'] == nav_stat_mapping['Under way sailing']) |
    (raw_df['NAVSTAT'] == nav_stat_mapping['Moored']) |
    (raw_df['NAVSTAT'].isna())
]

# We will groupby day also, so that we do not have huge gaps to interpolate
raw_df['DATE TIME (UTC)'] = pd.to_datetime(raw_df['DATE TIME (UTC)'])
raw_df['day'] = raw_df['DATE TIME (UTC)'].dt.date

grouped_df = raw_df\
    .set_index('DATE TIME (UTC)')\
    .groupby(['MMSI', 'day'])

dfs = []
all_time_diffs = []
missing_count = 0
above_interpolation_limit = 0
mmsi_timestamps = {}

for (mmsi, day), group_df in grouped_df:

    group_df = group_df\
        [['LATITUDE', 'LONGITUDE', 'SPEED', 'COURSE', 'HEADING']]\
        .drop_duplicates()

    # We define a track to be at least 50 measurements,
    #   and does not have a gap bigger than 2 hours
    time_diffs = group_df.reset_index()\
        ['DATE TIME (UTC)']\
        .diff(1)

    time_diffs = time_diffs / timedelta(minutes=1)
    all_time_diffs.extend( list(time_diffs) )

    # Skip, if the time difference is too big
    max_time_diff = max(all_time_diffs)

    # interpolate
    group_df = group_df[['LATITUDE', 'LONGITUDE', 'SPEED', 'COURSE', 'HEADING']]\
        .resample(f'300s')\
        .first()

    group_df['MMSI'] = mmsi
    group_df['day'] = day

    # At least 50, bcause input - 25 minutes, output - 25 minutes
    if group_df.dropna().shape[0] < 10:
        continue

    missing_count += np.sum( group_df.isnull() )
    group_df = group_df.dropna().reset_index()

    # Source: https://stackoverflow.com/questions/28773342/truncate-timestamp-column-to-hour-precision-in-pandas-dataframe
    group_df['hour'] = group_df['DATE TIME (UTC)'].dt.round('h')

    # merge with meteorological data
    group_df = group_df.apply(
        lambda row: merge_with_meteo(row, meteo_df, lat_lon_grid, lat_lon_tree),
        axis='columns'
    )
    group_df = group_df.drop(columns=['hour'])

    # create features such as dif and delta dif
    group_df, timestamps = feature_engineering(group_df)

    # Pasalinti NA ir atnaujinti timestamps pagal tai
    na_mask = group_df.isna().any(axis=1)

    group_df = group_df.loc[~na_mask, :].copy()
    timestamps = timestamps[~na_mask].copy()

    mmsi_timestamps[ (mmsi, day) ] = timestamps

    group_df['MMSI'] = int(mmsi)
    group_df['day'] = day
    dfs.append( group_df )

df = pd.concat( dfs, ignore_index=True )

min_max_scale_cols = [
    'SPEED',
    'currentSpeed',
    'gust',
    'swellHeight',
    'waveHeight',
    'windSpeed',
]

scaler = MinMaxScaler(feature_range=(-1, 1))

df[min_max_scale_cols] = scaler.fit_transform(
    df[min_max_scale_cols].values
)

In [7]:
def prepare_time_series(
    df: pd.DataFrame,
    n_lags: int = STEP_BACK_DEFAULT,
    n_future_lag: int = STEP_FORW_DEFAULT
):
    outputs = [ 'COURSE-dif', 'delta_dif', 'dif' ]
    n_features = df.shape[1]

    # so the input would be in accordance with:
    # https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM#call-arguments
    X_arr = np.empty( (df.shape[0], n_lags, n_features) )
    y_arr = np.empty( (df.shape[0], n_future_lag, len(outputs)) )

    for ix, col in enumerate( df.columns ):
        for lag in range( 1, n_lags+1 ):
            X_arr[:, lag-1, ix] = df[col].shift(-1 * lag)

    for ix, col in enumerate( outputs ):
        for lag in range( 1, n_lags+1 ):
            y_arr[:, lag-1, ix] = df[col].shift(lag)

    return X_arr[n_lags:-n_lags, :, :], y_arr[n_lags:-n_lags, :, :]

data = {}

for mmsi, mmsi_df in df.groupby('MMSI'):

    na_mask = mmsi_df.isna().any(axis=1)
    mmsi_df = mmsi_df.loc[~na_mask, :].copy()
    mmsi_df.drop( columns=['MMSI', 'day'], inplace=True )

    mmsi_df = mmsi_df\
        .rolling(ROLLING_WINDOW_SIZE)\
        .mean()

    #  Isimt NA vel "for good measure"
    na_mask = mmsi_df.isna().any(axis=1)
    mmsi_df = mmsi_df.loc[~na_mask, :].copy()

    X_arr, y_arr = prepare_time_series(mmsi_df)
    data[mmsi] = (X_arr, y_arr)

In [8]:
eagles_x, eagles_y = data[518998865]

eagles_y = eagles_y.reshape((-1, SLIDINGW_STEPS*N_OUT_FEATS))
eagles_x = eagles_x.reshape((-1, SLIDINGW_STEPS*N_INP_FEATS))

In [9]:
eagles_x.dump('../../output/data_eagles_vezhen/lstm-ae/eagles_x.npy')
eagles_y.dump('../../output/data_eagles_vezhen/lstm-ae/eagles_y.npy')

In [10]:
vezhen_x, vezhen_y = data[229659000]

vezhen_y = vezhen_y.reshape((-1, SLIDINGW_STEPS*N_OUT_FEATS))
vezhen_x = vezhen_x.reshape((-1, SLIDINGW_STEPS*N_INP_FEATS))

In [11]:
vezhen_x.dump('../../output/data_eagles_vezhen/lstm-ae/vezhen_x.npy')
vezhen_y.dump('../../output/data_eagles_vezhen/lstm-ae/vezhen_y.npy')